In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from calculator import Calculator

In [9]:
calc = Calculator(num1=0.0, num2=0.0)
calc

Calculator(num1=0.0, num2=0.0)

In [ ]:
print(calc.add(1, 2))
print(calc.subtract(5, 3))
print(calc.multiply(4, 6))
print(calc.get_current_datetime())

3
2
24
2026-05-21T20:14:55.939176


In [28]:
params = {"command": "uv", "args": ["run", "calc_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools() # just by running this, it is going to create an MCP Client and it is going to spawn our MCP Server by carrying out this instruction in 'params' and it is going to ask what tools do you offer us

mcp_tools

[Tool(name='get_current_dateTime', title=None, description='Get the current date and time.\n\n    Returns:\n        The current date and time as an ISO string.\n    ', inputSchema={'properties': {}, 'title': 'get_current_dateTimeArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_current_dateTimeOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='add', title=None, description='Add two numbers.\n\n    Args:\n        num1: The first number\n        num2: The second number\n\n    Returns:\n        The sum of the two numbers\n    ', inputSchema={'properties': {'num1': {'title': 'Num1', 'type': 'number'}, 'num2': {'title': 'Num2', 'type': 'number'}}, 'required': ['num1', 'num2'], 'title': 'addArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'addOutput', 'type': 'object'}, ic

In [29]:
instructions = "You are able to perform basic arithmetic operations such as addition, subtraction, multiplication, and division. You can also provide the current date and time. Please use the provided tools to answer the user's request."
# Explicitly instruct the agent exactly which tools to execute
request = (
    "Use your 'multiply' and 'add' tools to calculate (456 * 23) + 789. "
    "Then use your 'get_current_datetime' tool to fetch the time. "
    "Once you get the outputs from these three tool calls, provide your final response and stop."
)
model = "gpt-4o-mini"

In [32]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="calc_agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("calc_agent_trace.json"):
        result = await Runner.run(
            agent, 
            request,
            max_turns=10 
        )
    display(Markdown(result.final_output))


The calculation \((456 * 23) + 789\) results in **11277**.

The current date and time is **2026-05-21T20:39:28**.